# Keep sub-samples of bigger Paloma trees, add Paloma/human sequences

Author: Alexander Maksiaev

Purpose: Re-create subsampled Paloma trees, this time including Paloma and human sequences

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [2]:
# Directory paths

home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
og_fastas = home + "Other/Paloma/Alignments/" 
keep_headers = home + "Other/Paloma/Trees/subsampled_keep/"

os.chdir(keep_headers)



In [3]:
segments = ["PB2", "PB1", "PA", "HA_H1", "HA_H3", "NP", "NA_N1", "NA_N2", "MP", "NS"]
kept_segs = {}

for dirpath, dirs, files in os.walk(keep_headers):
    for file in files:
        file_name = os.path.join(dirpath, file)
        txt_name = file_name.split("/")[-1]
        for segment in segments:
            if segment in txt_name:
                kept_segs[segment] = file_name

In [4]:


for dirpath, dirs, files in os.walk(og_fastas):
    for file in files:
        file_name = os.path.join(dirpath, file)
        fasta_name = file_name.split("/")[-1]
        for segment in segments:
            if segment in file_name and "trimmed_deduplicated" in file_name and segment in kept_segs.keys():
                ids_to_keep = []
                df = df_from_fasta(file_name)
                
                with open(kept_segs[segment]) as kept_segs_txt:
                    for line in kept_segs_txt.readlines():
                        id = line.split("|")[0]
                        ids_to_keep.append(id)
                    kept_segs_txt.close()
                print(len(ids_to_keep))
                humans = list(df[df["full_header"].str.contains("human")]["full_header"].apply(lambda x: x.split("|")[0][1:])) # Get IDs, minus indicator ">"
                print(len(humans))
                paloma = list(df[df["full_header"].str.contains("Iberian|White", regex=True)]["full_header"].apply(lambda x: x.split("|")[0][1:]))
                print(len(paloma))
                ids_to_keep = ids_to_keep + humans + paloma
                print(len(ids_to_keep))
                df["id"] = df["full_header"].apply(lambda x: x.split("|")[0][1:]) 
                final_df = df[df["id"].isin(ids_to_keep)]
                df_to_fasta(final_df, segment + "_parnas_humans_Paloma.fasta", keep_headers)
    break

100
3
42
145
100
0
2
102
1
3
45
49
1
3
19
23
1
0
25
26
1
3
45
49
1
3
45
49
1
3
43
47
100
3
44
147
1
3
45
49
